# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an interactive exploration of the [FAIR^2 clinical oncology dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Obtain metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each record set, field, and column is identified by an `@id`. We'll print all record sets and their key fields with their `@id`.


In [ ]:
# List all record sets in the dataset and the fields inside
record_sets = list(dataset.record_sets)

if len(record_sets) == 0:
    print("No record sets defined in the root metadata; attempting to discover from package.")
    # As fallback: Croissant schema may not set top-level recordSet, so list all record sets via dataset API
    # Retrieve all record sets and print them
    all_record_sets = [r for r in dataset.record_sets]
    if all_record_sets:
        for rs in all_record_sets:
            print(f"RecordSet name: {rs.name} | @id: {rs.id}")
            print("  Fields:")
            for field in rs.fields:
                print(f"    Field: {field.name} | @id: {field.id} | dataType: {getattr(field, 'data_type', None)}")
    else:
        print("Could not find any record sets in the dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name} | @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field: {field.name} | @id: {field.id} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all record set `@id`s, and demonstrate loading as DataFrames. Adjust the variables if the dataset structure changes.


In [ ]:
# Find all record set ids
record_sets = [r for r in dataset.record_sets]
record_set_ids = [r.id for r in record_sets]

print(f"Record set @ids: {record_set_ids}")

dataframes = {}
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Loaded DataFrame for record set {rs.id} with shape: {df.shape}")
        else:
            print(f"No records found for record set {rs.id}")
    except Exception as e:
        print(f"Error loading data for record set {rs.id}: {e}")

# For demonstration, select the first record set id with data
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing {main_record_set_id} as primary table.")
    print("Columns (field @ids):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set data could be loaded.")


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, including filtering and normalization, on selected numeric and categorical fields.

Below, we select a numeric field by browsing DataFrame columns, then filter, normalize, and group by a categorical field, referencing all by their `@id`.

In [ ]:
# --- Setup: choose numeric and group fields by inspecting the DataFrame ---
df = dataframes[main_record_set_id]

# Try to auto-select a numeric field, or fall back to manual choice
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
if len(numeric_candidates) == 0:
    print("No numeric fields detected. Selecting field by pattern matching...")
    # Try to find likely numeric columns (e.g., Age, Interval)
    for col in df.columns:
        if any(substr in col.lower() for substr in ['age', 'interval', 'year', 'months', 'days']):
            numeric_candidates.append(col)
    if numeric_candidates:
        print(f"Candidates: {numeric_candidates}")
else:
    print(f"Numeric candidates: {numeric_candidates}")

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Set as None to prevent error; EDA will be skipped
    numeric_field_id = None

# Try to select a group (categorical) field, e.g., Sex, MSI status, etc.
group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
if len(group_field_candidates) > 0:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = None

# --- Apply filtering and normalization if numeric field exists ---
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    print(f"\nFiltering records where '{numeric_field_id}' > {threshold:.2f}")
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records shape: {filtered_df.shape}")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # --- Grouping ---
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print("No suitable group field available for grouping.")
else:
    print("No numeric field found, skipping numeric EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping possible
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR⁲ dataset on second primary colorectal cancer survivors using the `mlcroissant` library. We demonstrated how to:

- Programmatically load a Croissant-compliant dataset using only its schema URL.
- Inventory the available record sets, fields, and their unique `@id`s.
- Load record sets into Pandas DataFrames and examine contents by `@id` fields.
- Apply exploratory analysis by filtering, normalizing, and grouping using Croissant field identifiers.
- Visualize numeric data distributions and group comparisons.

Further analysis could include more detailed statistical summaries or model building, all using the `mlcroissant` library to ensure robust, schema-driven data handling.